# 03 - Results Analysis

Analyze trained model results: confusion matrices, hypnogram comparisons,
cross-dataset generalization, and comparison against baselines.

**Steps:**
1. Load model results and evaluation metrics
2. Confusion matrix heatmaps
3. Per-class precision/recall/F1
4. Hypnogram comparisons (predicted vs ground truth)
5. Cross-dataset generalization analysis
6. Comparison against published baselines

In [ ]:
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score,
    f1_score, accuracy_score
)

sns.set_theme(style="whitegrid")
%matplotlib inline

PROJECT_DIR = Path("/data/priorj/sleep_stage_classification")
RESULTS_DIR = PROJECT_DIR / "results"

STAGE_NAMES_5 = ["WAKE", "N1", "N2", "N3", "REM"]
STAGE_NAMES_3 = ["WAKE", "NREM", "REM"]
STAGE_NAMES_2 = ["WAKE", "SLEEP"]

## 1. Load Results

In [ ]:
# List available experiment results
result_dirs = sorted([d for d in RESULTS_DIR.iterdir() if d.is_dir()])
print("Available experiment results:")
for d in result_dirs:
    print(f"  {d.name}")

In [ ]:
def load_experiment_results(exp_name):
    """Load results for an experiment."""
    exp_dir = RESULTS_DIR / exp_name
    results = {"name": exp_name}
    
    # Load training history if available
    history_path = exp_dir / "history.json"
    if history_path.exists():
        with open(history_path) as f:
            results["history"] = json.load(f)
    
    # Load evaluation metrics
    metrics_path = exp_dir / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            results["metrics"] = json.load(f)
    
    # Load config
    config_path = exp_dir / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            results["config"] = json.load(f)
    
    return results

# Load primary 5-class results
primary_exp = "wearable-ss-5-tcn"
if (RESULTS_DIR / primary_exp).exists():
    results_5class = load_experiment_results(primary_exp)
    print(f"Loaded results for {primary_exp}")
else:
    print(f"Results for {primary_exp} not yet available.")
    print("Run training first: qsub scripts/train_model.sh")
    results_5class = None

## 2. Training Curves

In [ ]:
def plot_training_curves(results):
    """Plot loss and accuracy training curves."""
    if "history" not in results:
        print("No training history available.")
        return
    
    history = results["history"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    if "loss" in history:
        axes[0].plot(history["loss"], label="Train Loss")
    if "val_loss" in history:
        axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title(f"{results['name']} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    
    # Accuracy
    acc_key = next((k for k in history if "acc" in k.lower() and "val" not in k), None)
    val_acc_key = next((k for k in history if "acc" in k.lower() and "val" in k), None)
    if acc_key:
        axes[1].plot(history[acc_key], label="Train Accuracy")
    if val_acc_key:
        axes[1].plot(history[val_acc_key], label="Val Accuracy")
    axes[1].set_title(f"{results['name']} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

if results_5class:
    plot_training_curves(results_5class)

## 3. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    """Plot normalized and raw confusion matrices side by side."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], vmin=0, vmax=1)
    axes[0].set_title(f"{title} (Normalized)")
    axes[0].set_ylabel("True")
    axes[0].set_xlabel("Predicted")
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1])
    axes[1].set_title(f"{title} (Counts)")
    axes[1].set_ylabel("True")
    axes[1].set_xlabel("Predicted")
    
    plt.tight_layout()
    plt.show()
    
    # Print metrics
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    print(f"Overall Accuracy: {accuracy_score(y_true, y_pred):.3f}")
    print(f"Cohen's Kappa: {cohen_kappa_score(y_true, y_pred):.3f}")
    print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.3f}")
    print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.3f}")

# NOTE: In practice, load y_true and y_pred from the evaluation output
# This is a placeholder - replace with actual predictions when available
print("Run evaluation to generate predictions, then reload this cell.")
print("Expected: sleepkit --task stage --mode evaluate --config configs/wearable-ss-5-tcn.json")

## 4. Hypnogram Comparison (Predicted vs Ground Truth)

In [ ]:
def plot_hypnogram_comparison(y_true, y_pred, class_names, title="", max_epochs=1000):
    """Plot predicted vs ground truth hypnogram."""
    n = min(len(y_true), max_epochs)
    times = np.arange(n) * 30 / 3600  # Convert 30-sec epochs to hours
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
    
    colors = ["#e74c3c", "#f39c12", "#3498db", "#2c3e50", "#2ecc71"]
    
    for ax, data, label in [(axes[0], y_true[:n], "Ground Truth"),
                             (axes[1], y_pred[:n], "Predicted")]:
        for stage_idx, name in enumerate(class_names):
            mask = data == stage_idx
            if np.any(mask):
                ax.scatter(times[mask], data[mask], c=colors[stage_idx],
                          s=3, label=name, alpha=0.8)
        ax.set_yticks(range(len(class_names)))
        ax.set_yticklabels(class_names)
        ax.invert_yaxis()
        ax.set_ylabel(label)
        ax.legend(loc="upper right", markerscale=4)
    
    axes[1].set_xlabel("Time (hours)")
    if title:
        axes[0].set_title(title)
    
    plt.tight_layout()
    plt.show()

print("Hypnogram comparison function ready.")
print("Provide y_true and y_pred arrays after model evaluation.")

## 5. Cross-Experiment Comparison

In [ ]:
# Compare results across experiments
experiments = [
    ("wearable-ss-2-tcn", 2, STAGE_NAMES_2),
    ("wearable-ss-3-tcn", 3, STAGE_NAMES_3),
    ("wearable-ss-5-tcn", 5, STAGE_NAMES_5),
]

comparison = []
for exp_name, n_classes, names in experiments:
    exp_dir = RESULTS_DIR / exp_name
    metrics_path = exp_dir / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
        comparison.append({
            "Experiment": exp_name,
            "Classes": n_classes,
            "Accuracy": metrics.get("accuracy", "N/A"),
            "Weighted F1": metrics.get("weighted_f1", "N/A"),
            "Kappa": metrics.get("kappa", "N/A"),
        })

if comparison:
    comp_df = pd.DataFrame(comparison)
    display(comp_df)
else:
    print("No experiment results found yet. Run training/evaluation first.")

## 6. Comparison Against Published Baselines

In [ ]:
# Published SleepKit baselines (from model zoo)
baselines = pd.DataFrame([
    {"Model": "SS-2-TCN-SM (SleepKit)", "Classes": 2, "Dataset": "MESA", "Accuracy": 88.9},
    {"Model": "SS-3-TCN-SM (SleepKit)", "Classes": 3, "Dataset": "MESA", "Accuracy": 84.4},
    {"Model": "SS-4-TCN-SM (SleepKit)", "Classes": 4, "Dataset": "MESA", "Accuracy": 75.5},
    {"Model": "SS-5-TCN-SM (SleepKit)", "Classes": 5, "Dataset": "MESA", "Accuracy": 68.4},
])

print("Published baselines (SleepKit model zoo):")
display(baselines)

print("\nOur targets:")
print("  2-class (Wake/Sleep): >85% accuracy (sanity check)")
print("  5-class (AASM): match or exceed 68% (multi-dataset advantage)")
print("  Key: Multi-dataset (MESA+STAGES) should improve generalization")

In [ ]:
# Bar chart: our results vs baselines
if comparison:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(baselines))
    width = 0.35
    
    ax.bar(x - width/2, baselines["Accuracy"], width, label="SleepKit Baseline", color="#95a5a6")
    
    # Match our results to baseline class counts
    our_acc = []
    for _, row in baselines.iterrows():
        match = next((c for c in comparison if c["Classes"] == row["Classes"]), None)
        our_acc.append(float(match["Accuracy"]) * 100 if match and match["Accuracy"] != "N/A" else 0)
    
    ax.bar(x + width/2, our_acc, width, label="Ours (MESA+STAGES)", color="#3498db")
    
    ax.set_xlabel("Number of Classes")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Our Results vs SleepKit Baselines")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{int(n)}-class" for n in baselines["Classes"]])
    ax.legend()
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()

## Summary

Key findings:
- 5-class accuracy vs 68% baseline
- Which stages are most confused (typically N1 <-> N2)
- Cross-dataset generalization improvement from multi-dataset training
- Feature set comparison (FS-W-PA-14 vs FS-WEARABLE-EXT)

**Next steps**: Edge export (Phase 5) or additional ablation experiments.